# ETF AI ハンズオン
## Part 1: データ探索 & Cortex AI Functions

このノートブックでは、ポートフォリオマネージャーの業務を想定した **データ探索** と
**Snowflake Cortex AI Functions** の活用方法を学びます。

### このパートで体験できること

1. **データ確認**: ETFポートフォリオデータの構造を把握する
2. **AI_SUMMARIZE**: マーケットニュースを自動要約する
3. **AI_SENTIMENT**: ニュースのセンチメント（強気/弱気）を分析する
4. **AI_CLASSIFY**: ニュースを投資判断カテゴリに自動分類する
5. **AI_EXTRACT**: ETFドキュメントから重要情報を構造化して抽出する

### 🚀 体験ポイント

> **「毎朝の情報収集が10分から10秒に！」**
>
> 10本のニュース記事を読んで市場動向を把握する作業が、
> SQLを1行書くだけで完了します。

### 前提条件
- `setup.sql` の実行が完了していること
- ウェアハウス: `DEMO_WH`
- データベース: `ETF_AI_HANDSON_DB`

> ⏱️ **このパートの目安時間: 35分**

In [ ]:
USE DATABASE ETF_AI_HANDSON_DB;
USE SCHEMA DEMO_SCHEMA;
USE WAREHOUSE DEMO_WH;
SELECT
    CURRENT_DATABASE()  AS DB,
    CURRENT_SCHEMA()    AS SCHEMA,
    CURRENT_WAREHOUSE() AS WH,
    CURRENT_ROLE()      AS ROLE;

## 1. データ探索

まず、setup.sql で作成されたデータの構造を確認します。

> ⏱️ **目安: 10分**

In [ ]:
-- テーブル一覧とレコード数を確認
SELECT
    TABLE_NAME  AS "テーブル名",
    ROW_COUNT   AS "レコード数",
    COMMENT     AS "説明"
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'DEMO_SCHEMA'
  AND TABLE_TYPE = 'BASE TABLE'
ORDER BY TABLE_NAME;

In [ ]:
-- ETF銘柄一覧（ETF運用会社 のラインナップ）
SELECT
    ETF_CODE                AS "コード",
    ETF_NAME_SHORT          AS "略称",
    CATEGORY                AS "カテゴリ",
    ASSET_CLASS             AS "資産クラス",
    TO_CHAR(AUM_BILLION_JPY, '9,999.99') || '億円' AS "純資産",
    TO_CHAR(NAV, '999,999') || '円'  AS "基準価額(100口)",
    TO_CHAR(EXPENSE_RATIO * 100, '0.0000') || '%' AS "コスト(年率)",
    CASE WHEN NISA_ELIGIBLE THEN '○' ELSE '-' END AS "NISA"
FROM DIM_ETF
ORDER BY AUM_BILLION_JPY DESC;

In [ ]:
-- ポートフォリオ一覧と運用概要
SELECT
    PORTFOLIO_ID    AS "ID",
    PORTFOLIO_NAME  AS "ポートフォリオ名",
    MANAGER_NAME    AS "PM",
    RISK_PROFILE    AS "リスクプロファイル",
    TARGET_RETURN   || '%' AS "目標リターン(年率)",
    TO_CHAR(AUM_MILLION_JPY / 100, '99,999.0') || '億円' AS "運用資産"
FROM DIM_PORTFOLIO
ORDER BY AUM_MILLION_JPY DESC;

In [ ]:
-- ポートフォリオ保有明細（P004: グローバル分散成長の例）
SELECT
    h.PORTFOLIO_ID   AS "ポートフォリオ",
    e.ETF_NAME_SHORT AS "ETF名称",
    e.ETF_CODE       AS "コード",
    e.CATEGORY       AS "カテゴリ",
    TO_CHAR(h.WEIGHT_PCT, '99.999') || '%'  AS "比重",
    TO_CHAR(h.MARKET_VALUE_JPY / 100000000, '99,999.0') || '億円' AS "時価評価額",
    TO_CHAR(h.UNREALIZED_GAIN_JPY / 100000000, '9,999.0') || '億円' AS "含み損益",
    TO_CHAR(h.UNREALIZED_GAIN_PCT, '99.0') || '%' AS "含み損益率"
FROM FACT_HOLDINGS h
JOIN DIM_ETF e ON h.ETF_CODE = e.ETF_CODE
WHERE h.PORTFOLIO_ID = 'P004'
ORDER BY h.WEIGHT_PCT DESC;

In [ ]:
-- 各ポートフォリオの2025年通年パフォーマンス比較
SELECT
    p.PORTFOLIO_ID    AS "ID",
    dp.PORTFOLIO_NAME AS "ポートフォリオ名",
    TO_CHAR(p.RETURN_1Y_PCT,  '99.0') || '%' AS "1年リターン",
    TO_CHAR(p.VOLATILITY_1Y,  '99.0') || '%' AS "ボラティリティ",
    TO_CHAR(p.SHARPE_RATIO,   '9.00')        AS "シャープレシオ",
    TO_CHAR(p.MAX_DRAWDOWN,   '99.0') || '%' AS "最大ドローダウン"
FROM FACT_PERFORMANCE p
JOIN DIM_PORTFOLIO dp ON p.PORTFOLIO_ID = dp.PORTFOLIO_ID
WHERE p.PERF_DATE = '2025-12-31'
  AND p.ETF_CODE IS NULL
ORDER BY p.RETURN_1Y_PCT DESC;

## 2. Cortex AI Functions の活用

### Cortex AI Functions とは？

Snowflake Cortex AI Functions は、SQL の中から直接呼び出せる AI 機能です。
ローカルへの AI モデルのデプロイや API キーの管理は **一切不要** です。

| 関数 | 用途 | 活用例 |
|---|---|---|
| `AI_SUMMARIZE` | テキスト要約 | ニュース記事を3行に圧縮 |
| `AI_SENTIMENT` | 感情分析 | 強気/弱気トーンの数値化 |
| `AI_CLASSIFY` | テキスト分類 | 買い材料/売り材料の自動判定 |
| `AI_EXTRACT` | 情報抽出 | ファクトシートの構造化データ化 |
| `AI_AGG` | 集約質問応答 | 全ニュースからの市場サマリー生成 |

> ⏱️ **目安: 25分**

### 2-1. AI_SUMMARIZE（ニュース自動要約）

`AI_SUMMARIZE(text)` で長文を自動的に短く要約します。
毎朝の情報収集作業を大幅に効率化できます。

In [ ]:
-- ニュース本文を自動要約（1〜2文に圧縮）
SELECT
    NEWS_DATE   AS "日付",
    HEADLINE    AS "ヘッドライン",
    AI_SUMMARIZE(BODY) AS "AI要約（自動生成）"
FROM MARKET_NEWS
ORDER BY NEWS_DATE DESC
LIMIT 5;

### 2-2. AI_SENTIMENT（センチメント分析）

`AI_SENTIMENT(text)` でテキストのポジティブ/ネガティブを数値で返します。
- **+1.0 に近い**: 強いポジティブ（買い材料）
- **-1.0 に近い**: 強いネガティブ（売り材料）
- **0 に近い**: ニュートラル

In [ ]:
-- ニュースのセンチメント分析（全記事スコアリング）
SELECT
    NEWS_DATE   AS "日付",
    HEADLINE    AS "ヘッドライン",
    ROUND(AI_SENTIMENT(BODY), 2) AS "スコア(-1〜+1)",
    CASE
        WHEN AI_SENTIMENT(BODY) >= 0.3  THEN 'ポジティブ (買い材料)'
        WHEN AI_SENTIMENT(BODY) <= -0.3 THEN 'ネガティブ (売り材料)'
        ELSE 'ニュートラル (様子見)'
    END AS "判定",
    RELEVANT_ETFS AS "関連ETF"
FROM MARKET_NEWS
ORDER BY AI_SENTIMENT(BODY) DESC;

In [ ]:
-- センチメント分析結果をテーブルに永続化
UPDATE MARKET_NEWS
SET SENTIMENT = CASE
    WHEN AI_SENTIMENT(BODY) >= 0.3  THEN 'ポジティブ'
    WHEN AI_SENTIMENT(BODY) <= -0.3 THEN 'ネガティブ'
    ELSE 'ニュートラル'
END;

-- 結果確認
SELECT
    SENTIMENT       AS "センチメント",
    COUNT(*)        AS "件数",
    LISTAGG(NEWS_ID, ', ') WITHIN GROUP (ORDER BY NEWS_DATE) AS "ニュースID"
FROM MARKET_NEWS
GROUP BY SENTIMENT
ORDER BY 件数 DESC;

### 2-3. AI_CLASSIFY（テキスト自動分類）

`AI_CLASSIFY(text, [categories])` でテキストを指定カテゴリに自動分類します。
ETFへの影響度別にニュースを振り分ける例を示します。

In [ ]:
-- ニュースを投資判断カテゴリに自動分類（確信度スコア付き）
SELECT
    NEWS_DATE   AS "日付",
    HEADLINE    AS "ヘッドライン",
    AI_CLASSIFY(
        HEADLINE || ' ' || BODY,
        ['買い材料', '様子見', '売り材料']
    ):label::VARCHAR AS "投資判断",
    ROUND(AI_CLASSIFY(
        HEADLINE || ' ' || BODY,
        ['買い材料', '様子見', '売り材料']
    ):score::FLOAT, 2) AS "確信度",
    RELEVANT_ETFS AS "関連ETF"
FROM MARKET_NEWS
ORDER BY NEWS_DATE DESC;

### 2-4. AI_EXTRACT（ETFドキュメントからの情報抽出）

`AI_EXTRACT(text, schema)` で非構造化テキストから指定した情報を構造化して抽出します。
ETFのファクトシートや目論見書から重要指標を自動抽出する例を示します。

In [ ]:
-- ETFファクトシートの概要セクションから重要指標を自動抽出
SELECT
    ETF_CODE    AS "コード",
    ETF_NAME    AS "ETF名称",
    AI_EXTRACT(
        CONTENT,
        {
            'pure_asset_amount':  'string',
            'expense_ratio':      'string',
            'annual_return_2025': 'string',
            'distribution_yield': 'string',
            'nisa_eligible':      'boolean'
        }
    ) AS "抽出された指標"
FROM ETF_DESCRIPTIONS
WHERE SECTION = 'ファンド概要'
ORDER BY ETF_CODE;

In [ ]:
-- リスク情報セクションから主要リスク項目を配列として抽出
SELECT
    ETF_CODE    AS "コード",
    AI_EXTRACT(
        CONTENT,
        {
            'main_risk_types':    'array',
            'capital_guarantee':  'boolean'
        }
    ) AS "リスク構造情報"
FROM ETF_DESCRIPTIONS
WHERE SECTION = 'リスク情報'
ORDER BY ETF_CODE;

### 2-5. AI_AGG（複数テキストの統合サマリー生成）

`AI_AGG(text, question)` で複数行のテキストを集約しながら自然言語で質問に答えます。
全ニュースを横断して「今月の市場動向サマリー」を自動生成します。

In [ ]:
-- 全ニュースから今月のETF市場動向サマリーを自動生成
SELECT
    AI_AGG(
        HEADLINE || ': ' || BODY,
        'ETFのポートフォリオマネージャー向けに、以下のニュースから市場動向の要点を
        以下の形式でまとめてください。
        1. 最重要ニュース（上位3件）
        2. ポジティブ材料
        3. リスク・注意材料
        4. 今週の推奨アクション'
    ) AS "AI生成マーケットサマリー"
FROM MARKET_NEWS
WHERE NEWS_DATE >= '2026-03-01';

## まとめ

Part 1 完了！Cortex AI Functions の基本操作を体験しました。

### 更新されたデータ

| テーブル | 操作 | 内容 |
|---|---|---|
| `MARKET_NEWS` | UPDATE | AI_SENTIMENT によるセンチメントフラグ付与 |

### AI Functions の威力

- **AI_SUMMARIZE**: 2,000文字のニュース → 2行に圧縮、情報収集時間を 1/10 に
- **AI_SENTIMENT**: ポジネガトーンを数値化し、重要ニュースを自動フィルタリング
- **AI_CLASSIFY**: 投資判断ラベルの自動付与、アナリストの分類作業を自動化
- **AI_EXTRACT**: PDF・非構造化文書 → 構造化データ、手入力不要
- **AI_AGG**: 大量ドキュメントの横断要約、朝のブリーフィング資料を自動生成

### よくある質問

**Q: PDF のファクトシートを直接処理できますか？**
A: `AI_PARSE_DOCUMENT` 関数でステージ上の PDF を直接テキスト化できます。
   その後 `SPLIT_TEXT_MARKDOWN_HEADER` でチャンキングし、Cortex Search に登録します（Part 3 で体験）。

**Q: 日本語でも精度は高いですか？**
A: はい。Snowflake Cortex は日本語対応の LLM を使用しており、
   金融ドメインの日本語テキストでも高精度で動作します。

**次のステップ**: `part2_cortex_analyst.ipynb` で Semantic View と
Cortex Analyst（自然言語 to SQL）を体験してください。